### Fit detected lightcurves with Gaussian Process to show the feasibility of preprocessing.

In [ ]:
import pandas as pd
import numpy as np
import os
from pprint import pprint
from astropy.io import fits
import matplotlib.pyplot as plt
import matplotlib as mpl
import george
from george import kernels
from scipy.optimize import minimize

In [ ]:
def convert_fluxcal_to_mag(flux, err):
    """Convert fluxcal to magnitude and add magnitude error

    Args:
        df (pd.DataFrame): DataFrame with columns "FLUXCAL" and "FLUXCALERR"
    Returns:
        pd.DataFrame: DataFrame with added "magnitude" and "magnitude error"
    """
    # Convert fluxcal to magnitude
    flux[flux <= 0] = np.nan

    # Compute magnitude; where flux was <= 0, magnitude will be NaN
    mag = -2.5 * np.log10(flux) + 27.5
    magerr = 2.5 / np.log(10) * err / flux
    return mag, magerr

In [ ]:
injections = pd.read_csv("<BASE_DIR>/ML+GW+KN/dataset/O5_sim_bns/injections_final.csv")

In [ ]:
sim_id = 16
SIM_PATH = "<BASE_DIR>/SNANA/SNDATA_ROOT/SIM/LSST_KN_BNS"
phot_file = os.path.join(SIM_PATH, f"LSST_KN_BNS_{sim_id}", f"LSST_KN_BNS_{sim_id}_PHOT.FITS")
head_file = os.path.join(SIM_PATH, f"LSST_KN_BNS_{sim_id}", f"LSST_KN_BNS_{sim_id}_HEAD.FITS")
# 打开PHOT.fits文件
phot_hdul = fits.open(phot_file)
phot_data = phot_hdul[1].data

# 打开HEAD.fits文件，获取每个源的光变曲线索引分界
head_hdul = fits.open(head_file)
head_data = head_hdul[1].data
start_indices = head_data['PTROBS_MIN']
end_indices = head_data['PTROBS_MAX']

mjd_explode = injections['mjd_time'][np.where(injections['simulation_id']==sim_id)[0]].values
mjd_explode

In [ ]:
bands = ['LSST-u','LSST-g','LSST-r','LSST-i','LSST-z','LSST-y']
for i, (start, end) in enumerate(zip(start_indices, end_indices)):
    plt.figure(figsize=(8,6))
    peakmjd = head_data['PEAKMJD'][i]
    print(f'Plotting source {i+1},Merger MJD: {mjd_explode[0]} PEAKMJD: {peakmjd}, Index range: {start}-{end}')
    plt.axvline(x=peakmjd, color='k', linestyle='--', label='PEAKMJD')
    plt.axvline(x=mjd_explode[0], color='r', linestyle='--', label="Merger MJD")
    source_data = phot_data[start-1:end]  # FITS索引通常从1开始
    source_data = source_data[np.logical_and(source_data['MJD']>=mjd_explode[0]-5, source_data['MJD']<=mjd_explode[0]+15)]
    mjd = source_data['MJD']
    flux = source_data['FLUXCAL']
    fluxerr = source_data['FLUXCALERR']
    mag, magerr = convert_fluxcal_to_mag(flux=flux.copy(), err=fluxerr)
    for band in bands:
        band_mask = source_data['BAND'] == band
        # plt.errorbar(mjd[band_mask], mag[band_mask], yerr=magerr[band_mask], fmt='o', label=band)
        plt.errorbar(mjd[band_mask], flux[band_mask], yerr=fluxerr[band_mask], fmt='o', label=band)
        # plt.errorbar(mjd, flux, yerr=err, fmt='o', label=f'Source {i+1}')
        plt.xlabel('MJD', fontsize=18)
        plt.xticks(fontsize=14)
        plt.ylabel('FLUXCAL', fontsize=18)
        plt.yticks(fontsize=14)
        plt.title('Light Curves from SNANA simulation', fontsize=18)
        plt.legend(fontsize=12)

In [ ]:
source_data = phot_data[start_indices[1]-1:end_indices[1]]  # FITS索引通常从1开始
source_data = source_data[np.logical_and(source_data['MJD']>=mjd_explode[0]-5, source_data['MJD']<=mjd_explode[0]+15)]
t_obs = source_data['MJD']
y_obs = source_data['FLUXCAL']
y_err = source_data['FLUXCALERR']
w_obs = source_data['BAND']
wave_map = {
    'LSST-u': 3.650, 'LSST-g': 4.750, 'LSST-r': 6.200, 
    'LSST-i': 7.550, 'LSST-z': 8.700, 'LSST-Y': 9.800
}
w_obs = np.array([wave_map[band] for band in w_obs])


In [ ]:
# 示例：构建输入矩阵 X (N行, 2列) -> [时间, 波长]
# 建议：为了数值稳定性，建议将波长归一化 (例如除以 1000)
X_train = np.column_stack([t_obs, w_obs])

# ==========================================
# 2. 定义核函数 (2D Kernel)
# ==========================================
# 注意：george 的 metric 参数通常对应 "长度尺度的平方" (length_scale^2)

# A. 振幅核 (控制整体信号强弱)
# 使用 log_constant，便于优化器处理
var_init = np.var(y_obs)
k_amp = kernels.ConstantKernel(log_constant=np.log(var_init), ndim=2)

# B. 时间核 (作用于第 0 列)
# 初始猜测: 时间尺度为 10 天 -> metric = 10^2 = 100
k_time = kernels.Matern32Kernel(metric=10.0**2, ndim=2, axes=0)

# C. 波长核 (作用于第 1 列)
# 初始猜测: 波长相关尺度为 2000A -> metric = 2000^2
# 如果你对波长进行了归一化(如 /1000)，这里也要相应缩小 (如 metric=2.0^2)
k_wave = kernels.ExpSquaredKernel(metric=2.0**2, ndim=2, axes=1)

# D. 组合核 (乘积)
kernel = k_amp * k_time * k_wave

# ==========================================
# 3. 实例化 GP 并预计算
# ==========================================
gp = george.GP(kernel)

# 预计算 (Pre-compute) 矩阵分解
# 这一步非常快，用于检查矩阵是否正定
gp.compute(X_train, y_err)

print("初始 Log Likelihood:", gp.log_likelihood(y_obs))

# ==========================================
# 4. 优化超参数 (Fitting)
# ==========================================

def neg_ln_like(p):
    gp.set_parameter_vector(p)
    # 这里的 quiet=True 防止非正定报错中断，而是返回 -inf
    return -gp.log_likelihood(y_obs, quiet=True)

def grad_neg_ln_like(p):
    gp.set_parameter_vector(p)
    return -gp.grad_log_likelihood(y_obs, quiet=True)

# 运行优化 (使用 L-BFGS-B)
print("开始优化参数...")
result = minimize(
    neg_ln_like,
    gp.get_parameter_vector(),
    jac=grad_neg_ln_like, # george 支持解析梯度，速度很快
    method="L-BFGS-B"
)

# 更新 GP 为最佳参数
gp.set_parameter_vector(result.x)
print("优化成功:", result.success)
print("最佳参数:", result.x)

# ==========================================
# 5. 预测 (Predicting)
# ==========================================
# 假设我们要预测 g 波段 (有效波长 ~4750) 在 t_pred 时间段的光变

t_pred = np.linspace(min(t_obs), max(t_obs), 100)
target_wave = 4.750  # g-band

# 构建预测矩阵 X_pred
X_pred = np.column_stack([t_pred, np.full_like(t_pred, target_wave)])

# 预测均值和方差
mu, var = gp.predict(y_obs, X_pred, return_var=True)
std = np.sqrt(var)

# mu 即为拟合的光变曲线，std 为不确定度
plt.plot(t_pred, mu, 'b-', label='GP Prediction (g-band)')

In [ ]:
# 定义波段对应的颜色，符合天文绘图习惯
band_colors = {
    'LSST-u': 'purple', 'LSST-g': 'blue', 'LSST-r': 'green', 
    'LSST-i': 'orange', 'LSST-z': 'red', 'LSST-Y': 'darkred'
}

# 2. 设置绘图
plt.figure(figsize=(12, 8))
t_grid = np.linspace(t_obs.min() - 5, t_obs.max() + 5, 500)

# 3. 遍历波段进行预测和绘图
for band in ['LSST-u', 'LSST-g', 'LSST-r', 'LSST-i', 'LSST-z', 'LSST-Y']:
    w_target = wave_map[band]
    color = band_colors[band]
    
    # 构建该波段的预测输入矩阵 [Time, Wavelength]
    X_pred = np.column_stack([t_grid, np.full_like(t_grid, w_target)])
    
    # 使用 GP 进行预测
    # mu: 均值, var: 方差
    mu, var = gp.predict(y_obs, X_pred, return_var=True)
    std = np.sqrt(var)
    
    # 提取该波段的原始观测点用于对比
    # 注意：这里假设 w_obs 是原始数据中存储的波长
    mask = (w_obs == w_target)
    
    # 绘制观测点
    if np.any(mask):
        plt.errorbar(t_obs[mask], y_obs[mask], yerr=y_err[mask], 
                     fmt='o', color=color, label=f'Obs {band}', 
                     markersize=5, alpha=0.7, capsize=0)
    
    # 绘制 GP 拟合曲线及 1-sigma 置信区间
    plt.plot(t_grid, mu, color=color, lw=2, label=f'GP {band}')
    plt.fill_between(t_grid, mu - std, mu + std, color=color, alpha=0.15)

# 4. 界面修饰
plt.xlabel("Time (MJD or Days)", fontsize=14)
plt.ylabel("Flux / Magnitude", fontsize=14)
plt.title("Multi-band Gaussian Process Light Curve Fit (george)", fontsize=16)
plt.legend(loc='best', ncol=2, frameon=True)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Huge uncertainty